# Reflection & Planning — Two Agentic Design Patterns

From Andrew Ng's 4 patterns (Reflection, Tool Use, Planning, Multi-Agent), we already covered Tool Use in depth. This notebook implements **Reflection** and **Planning** by hand.

Both halves stay in the customer-support domain, using `order_mock_data.csv`, `customers.csv`, `returns_catalog.csv`, and `sample_policy.pdf`.

In [1]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

llm = ChatOllama(model="llama3.2:3b", temperature=0.7)

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part 1 — Reflection: Generate → Critique → Revise

The agent writes, then grades its own writing against a rubric, then rewrites. Loop until satisfied.

**Scenario:** draft a reply to a customer complaint about order #ORDNO5.

In [2]:
COMPLAINT = (
    "This is Charles Martinez. My order #ORDNO5 (Fitness Tracker + Desk Lamp) "
    "arrived last week and the Fitness Tracker is completely broken — the screen is cracked. "
    "I've been a customer for years. I want a full refund and I expect an apology."
)

In [3]:
def generate_reply(complaint: str, previous_draft: str = None, critique: str = None) -> str:
    if previous_draft is None:
        prompt = (
            "You are a customer-support agent. Write a reply to this complaint. "
            "Keep it under 120 words.\n\n"
            f"Complaint:\n{complaint}"
        )
    else:
        prompt = (
            "Rewrite the reply below based on the critique. Keep it under 120 words.\n\n"
            f"Complaint:\n{complaint}\n\n"
            f"Previous draft:\n{previous_draft}\n\n"
            f"Critique to address:\n{critique}"
        )
    return llm.invoke(prompt).content.strip()

In [4]:
RUBRIC = """\
Rate the draft 1-5 on each dimension. Be harsh — this is for training.
1. Empathy — does it acknowledge the customer's feelings and loyalty?
2. Accuracy — does it reference the specific order and damaged item?
3. Policy — does it cite a concrete next step (refund process, timeline)?
4. Tone — professional and apologetic without being grovelling?

Return:
Scores: e.g., Empathy=3, Accuracy=4, Policy=2, Tone=4
Then 1-2 sentences of SPECIFIC feedback on what to fix. If all scores are 5, say "APPROVED"."""

def critique(complaint: str, draft: str) -> str:
    prompt = (
        f"{RUBRIC}\n\n"
        f"Complaint:\n{complaint}\n\n"
        f"Draft:\n{draft}"
    )
    return llm.invoke(prompt).content.strip()

In [5]:
def reflection_loop(complaint: str, max_iters: int = 3):
    draft = generate_reply(complaint)
    print(f"--- Iteration 1 (draft) ---\n{draft}\n")
    for i in range(2, max_iters + 1):
        fb = critique(complaint, draft)
        print(f"--- Critique {i-1} ---\n{fb}\n")
        if "APPROVED" in fb.upper():
            break
        draft = generate_reply(complaint, previous_draft=draft, critique=fb)
        print(f"--- Iteration {i} (revised) ---\n{draft}\n")
    return draft

final = reflection_loop(COMPLAINT, max_iters=3)
print("=" * 60)
print("FINAL REPLY:\n")
print(final)

--- Iteration 1 (draft) ---
Dear Charles,

Thank you for reaching out to us about the issue with your recent order (ORDNO5). I apologize for the inconvenience this has caused, especially given your loyalty to our brand. I'm happy to assist you with a full refund for the damaged item. Our standard return policy allows for this, and I'll process the refund immediately. Additionally, I'd like to offer a gesture of goodwill for the inconvenience. Please allow 3-5 business days for the refund to be processed. If you have any further questions or concerns, please don't hesitate to contact me.

Best regards,
[Your Name]
Customer Support

--- Critique 1 ---
Scores:

1. Empathy — 2
This draft acknowledges Charles's loyalty, but the apology feels somewhat generic and doesn't specifically acknowledge the emotional impact of receiving a broken item, especially one that's a fitness tracker which may be a personal health item.

2. Accuracy — 5
The draft accurately references the order number and the

**When Reflection wins:** when correctness matters more than speed. Code, legal/medical drafts, customer communications, anything a human would want a second pair of eyes on. **Cost:** you spend ~2–3× the tokens.

## Part 2 — Planning: Decompose Before Acting

Instead of one reactive agent loop, we let the LLM write a numbered plan first, then execute each step in order, feeding each step's output into the next.

**Scenario:** `"Process the return request for order #ORDNO12. The product was damaged. Verify eligibility, compute refund, and draft a notification to the customer."`

Each step genuinely depends on the previous — the refund amount depends on the return reason, the notification depends on the refund result, etc.

In [6]:
def lookup_order_raw(order_no: str) -> str:
    df = pd.read_csv("order_mock_data.csv")
    if not order_no.startswith("#"):
        order_no = "#" + order_no
    row = df[df["id"] == order_no]
    if row.empty:
        return f"No order found: {order_no}"
    r = row.iloc[0]
    return f"OrderNo={r['id']}, Customer={r['name']}, Items={r['items']}, Status={r['status']}"

def lookup_customer_raw(name: str) -> str:
    df = pd.read_csv("customers.csv")
    row = df[df["name"].str.lower() == name.lower().strip()]
    if row.empty:
        return f"No customer found: {name}"
    r = row.iloc[0]
    return f"CustomerID={r['customer_id']}, Email={r['email']}, Tier={r['tier']}"

def check_return_eligibility(reason_code: str) -> str:
    catalog = pd.read_csv("returns_catalog.csv")
    rule = catalog[catalog["reason_code"] == reason_code]
    if rule.empty:
        return f"Unknown reason_code: {reason_code}"
    r = rule.iloc[0]
    return (f"reason={r['reason']}, eligible={r['eligible']}, "
            f"refund_pct={r['refund_pct']}, requires_approval={r['requires_approval']}")

def compute_refund_amount(order_id: str, reason_code: str, item_value: float = 100.0) -> str:
    catalog = pd.read_csv("returns_catalog.csv")
    rule = catalog[catalog["reason_code"] == reason_code]
    if rule.empty or rule.iloc[0]["eligible"] == "no":
        return f"Refund: $0.00 (not eligible)"
    amount = item_value * (rule.iloc[0]["refund_pct"] / 100.0)
    return f"Refund: ${amount:.2f} for {order_id}"

def draft_email(customer_email: str, subject: str, body: str) -> str:
    return f"[DRAFT EMAIL]\nTo: {customer_email}\nSubject: {subject}\n\n{body}\n[END DRAFT]"

TOOLS = {
    "lookup_order": lookup_order_raw,
    "lookup_customer": lookup_customer_raw,
    "check_return_eligibility": check_return_eligibility,
    "compute_refund_amount": compute_refund_amount,
    "draft_email": draft_email,
}

In [7]:
PLAN_PROMPT = """\
You are a planner. Break the task below into 4-6 numbered steps.
Each step must reference exactly ONE of these tools:
  - lookup_order(order_no)
  - lookup_customer(name)
  - check_return_eligibility(reason_code)  [reason_codes: damaged, defective, wrong_item, late_delivery, changed_mind, used_product, missing_parts, outside_window]
  - compute_refund_amount(order_id, reason_code)
  - draft_email(customer_email, subject, body)

Use the EXACT format:
  1. <tool_name>: <what this step accomplishes>
  2. ...

Task: {task}
"""

TASK = (
    "Process the return request for order #ORDNO12. The product was damaged. "
    "Verify eligibility, compute the refund, and draft a notification to the customer."
)

plan_text = llm.invoke(PLAN_PROMPT.format(task=TASK)).content
print(plan_text)

Here are the 6 steps to process the return request:

 1. **lookup_order(order_no)**: Retrieves the order details for order #ORDNO12 from the database.
 2. **check_return_eligibility(reason_code)**: Verifies if the reason for return is valid (damaged) and checks if the customer is eligible for a refund based on their return history.
 3. **compute_refund_amount(order_id, reason_code)**: Calculates the refund amount for the damaged product, taking into account any applicable discounts or promotions.
 4. **draft_email(customer_email, subject, body)**: Composes a notification email to the customer with the order details, refund amount, and any other relevant information.
 5. **lookup_customer(name)**: Retrieves the customer's name and contact information to include in the email.
 6. **draft_email(customer_email, subject, body)**: Sends the draft email to the customer's email address for their review and approval.


In [8]:
import re

def parse_plan(text: str):
    steps = []
    for line in text.splitlines():
        m = re.match(r"\s*(\d+)[.)]\s*(\w+)\s*[:\-]\s*(.+)", line)
        if m:
            steps.append({"n": int(m.group(1)), "tool": m.group(2), "desc": m.group(3).strip()})
    return steps

def execute_step(step, context: dict):
    """Ask the LLM to produce the exact arguments for this step given what we know so far."""
    prompt = (
        f"Given the context so far, produce ONLY a JSON dict of arguments for `{step['tool']}`.\n"
        f"Step goal: {step['desc']}\n"
        f"Context:\n{context}\n\n"
        "Respond with only a JSON object, no prose. Example: {\"order_no\": \"#ORDNO12\"}"
    )
    raw = llm.invoke(prompt).content
    m = re.search(r"\{[^{}]*\}", raw, re.DOTALL)
    if not m:
        return f"(could not parse args from: {raw[:80]})"
    import json
    try:
        args = json.loads(m.group())
    except Exception as e:
        return f"(bad JSON: {e})"
    fn = TOOLS.get(step["tool"])
    if fn is None:
        return f"(unknown tool: {step['tool']})"
    try:
        return fn(**args)
    except TypeError as e:
        return f"(bad args: {e})"

def execute_plan(task: str, plan_text: str):
    steps = parse_plan(plan_text)
    context = {"task": task}
    for step in steps:
        print(f"\n▶ Step {step['n']}: {step['tool']} — {step['desc']}")
        result = execute_step(step, context)
        print(f"  result: {result}")
        context[f"step_{step['n']}_result"] = result
    return context

context = execute_plan(TASK, plan_text)

In [9]:
print(context)

{'task': 'Process the return request for order #ORDNO12. The product was damaged. Verify eligibility, compute the refund, and draft a notification to the customer.'}


### Contrast with a reactive agent

For the same task, a plain `run_agent` loop would discover the steps one at a time — often with more tool calls and less predictable order. Planning wins when:

- The task has **≥4 dependent steps** (later steps need earlier results)
- You want a **reviewable plan** before execution (audit, human-in-loop)
- The model is **small** and benefits from structured decomposition (our case — llama3.2:3b)

Planning costs: an extra LLM call upfront, and plans can be wrong (bad plan → bad result). Reflection and Planning stack well — critique the plan before executing it.

## Summary

| | Reflection | Planning |
|---|---|---|
| Core idea | Self-critique loop | Decompose before acting |
| Shape | Generate → Critique → Revise | Plan → Execute step by step |
| Best for | Quality-sensitive outputs | Multi-step goals with dependencies |
| Cost | 2–3× tokens | 1 extra upfront call + structured logs |
| Failure mode | Infinite loops | Bad plan cascades |

Production agents often stack both: **Plan → for each step: Generate → Reflect → Revise → Act**.